# Rehab Coach RL: train all sweeps on Kaggle

In [ ]:
# Clone the repository
REPO_URL = "https://github.com/Tresorshingiro/Tresor_Nkurunziza_rl_summative.git"
!rm -rf /kaggle/working/repo
!git clone --depth 1 {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo

# Remove the short local test runs that were committed, so they are not counted
# as finished runs or picked as the "best" model by the evaluation.
!rm -f models/dqn/dqn_smoke.pt models/pg/*_default.pt logs/dqn/dqn_smoke.csv logs/*/*_default.csv
!ls training

In [ ]:
# Install TorchRL WITHOUT replacing Kaggle's PyTorch build.
# The constraints file pins torch to the installed version, so pip picks a matching torchrl.
import subprocess, sys, torch
installed = torch.__version__.split("+")[0]          # "2.6.0+cu124" -> "2.6.0"
with open("/kaggle/working/constraints.txt", "w") as f:
    f.write(f"torch=={installed}\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/kaggle/working/constraints.txt",
                "torchrl", "tensordict", "gymnasium>=1.0", "pygame", "PyOpenGL"], check=True)

In [ ]:
# Check versions and the GPU
import torch, torchrl, tensordict, gymnasium
print("torch", torch.__version__, "| torchrl", torchrl.__version__, "| tensordict", tensordict.__version__,
      "| gymnasium", gymnasium.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Benchmark: time a short DQN and PPO run on each device and project the total time.
import time
from training.dqn_training import train_dqn
from training.pg_training import train_pg

STEPS = 150_000        # per-run budget, identical for every config
BENCH = 8_192
devices = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
seconds = {}
for device in devices:
    start = time.time()
    train_dqn({"config_id": f"bench_dqn_{device}", "total_steps": BENCH}, device=device,
              out_root="/kaggle/working/bench", resume=False)
    seconds[("dqn", device)] = time.time() - start
    start = time.time()
    train_pg({"algo": "ppo", "config_id": f"bench_ppo_{device}", "total_steps": BENCH}, device=device,
             out_root="/kaggle/working/bench", resume=False)
    seconds[("ppo", device)] = time.time() - start
for (algo, device), s in seconds.items():
    print(f"{algo:<4} {device:<5} {s:6.1f}s for {BENCH} steps -> ~{12 * s * STEPS / BENCH / 3600:.1f} h for 12 runs")
DEVICE = min(devices, key=lambda d: seconds[("dqn", d)] + seconds[("ppo", d)])
print("Using device:", DEVICE)
print("The sweeps run in parallel, so the total time is roughly the DQN line above "
      "(a bit more, since 4 processes share the CPUs).")

In [ ]:
# (Second session only) restore finished runs from a previous version's output.
# Add that version's output as an input (Add Input -> Your Work), then run this cell.
import glob, zipfile
previous = glob.glob("/kaggle/input/**/results.zip", recursive=True)
if previous:
    zipfile.ZipFile(previous[0]).extractall(".")
    print("restored", previous[0])
else:
    print("no previous results found - starting fresh")

In [ ]:
# Run the sweeps: one background process per algorithm (4 vCPUs -> 4 processes).
# Finished runs are skipped and interrupted runs resume from their checkpoint.
import glob, os, subprocess, sys, time
ALGOS = ["dqn", "ppo", "a2c", "reinforce"]      # split across two sessions if needed
env = {**os.environ, "OMP_NUM_THREADS": "1"}    # stop 4 processes fighting over threads

def finished_runs():
    return {a: len(glob.glob(f"models/{'dqn' if a == 'dqn' else 'pg'}/{a}_[0-9][0-9].pt")) for a in ALGOS}

procs = {}
for algo in ALGOS:
    log = open(f"/kaggle/working/sweep_{algo}.log", "a")
    procs[algo] = subprocess.Popen([sys.executable, "-m", "training.sweeps", "--algo", algo,
                                    "--steps", str(STEPS), "--device", DEVICE],
                                   stdout=log, stderr=subprocess.STDOUT, env=env)
start = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(time.strftime("%H:%M"), f"{(time.time() - start) / 3600:.1f} h,", "finished runs (of 12):",
          finished_runs(), flush=True)
print("exit codes:", {a: p.returncode for a, p in procs.items()})
print("finished runs:", finished_runs())
!tail -n 5 /kaggle/working/sweep_*.log

In [ ]:
# Pick the best config per algorithm, test generalisation, draw the figures.
# Failures here do not stop the notebook, so the zip in the next cell is always made.
import os, subprocess, sys
if os.path.exists("training/evaluate.py") and os.path.exists("training/plots.py"):
    subprocess.run([sys.executable, "-m", "training.evaluate"])
    subprocess.run([sys.executable, "-m", "training.plots"])
else:
    print("training/evaluate.py or training/plots.py is not in the GitHub repo: "
          "run them locally after unzipping results.zip.")

In [ ]:
# Zip everything for download (Output tab -> results.zip)
!cp /kaggle/working/sweep_*.log logs/ 2>/dev/null || true
!zip -qr /kaggle/working/results.zip models logs assets
!ls -lh /kaggle/working/results.zip